# All-rat poster interpretation

Inspect the aggregate export for the fixed 500 ms comparison. No raw recordings or checkpoints are needed. See the [poster guide](../poster/README.md) for panel sources and regeneration.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

here = Path.cwd().resolve()
root = next(p for p in (here, *here.parents) if (p / 'poster/interpretation-all-rats.json').is_file())
evidence = json.loads((root / 'poster/interpretation-all-rats.json').read_text())
rats = ('Mitt', 'Barat', 'Stella', 'Superchris', 'Buchanan')
target = 'position'  # odor or position
class_index = 3     # 1–5 (A–E for odor)
by_rat = {r['rat']: r for r in evidence['weights'] if r['target'] == target}

## Amplitude + L1 weights and sparsity

Signed and absolute weights are class-centered, then summarized across channels within each rat. Sparsity is the fraction of fitted coefficients exactly zero before centering. Channel numbers are not treated as anatomically aligned.

In [ ]:
fig, axes = plt.subplots(3, len(rats), figsize=(14, 7), sharex=True, sharey=True, layout='constrained')
for i, (field, title, cmap) in enumerate((('signed', 'Mean signed weight', 'RdBu_r'),
                                        ('magnitude', 'Mean absolute weight', 'YlGn'),
                                        ('nonzero', 'Fraction zero', 'Greys'))):
    values = [np.array(by_rat[r][field][class_index-1]).T for r in rats]
    if field == 'nonzero':
        values = [1 - x/by_rat[r]['n_channels'] for r,x in zip(rats, values, strict=True)]
    vmax = 1 if field == 'nonzero' else max(np.abs(values).max(), 1e-12)
    for j, (ax, rat, x) in enumerate(zip(axes[i], rats, values, strict=True)):
        im = ax.imshow(x, origin='lower', extent=(0,500,0,12), aspect='auto', cmap=cmap,
                       vmin=-vmax if field=='signed' else 0, vmax=vmax)
        if i == 0:
            ax.set_title(rat)
        if j == 0:
            ax.set(yticks=[.5,11.5], yticklabels=['4 Hz','150 Hz'])
    fig.colorbar(im, ax=axes[i], label=title)
fig.supxlabel('Time after odor onset (ms)');

## Validation masking and CEBRA centroids

Masking shows masked minus intact validation balanced accuracy (percentage points), without refitting. Negative means worse after masking. PCA is fit separately on each training embedding. Centroid labels show the validation trial count. The OutSeq setting is only applicable to the position encoder.

In [ ]:
mask = pd.DataFrame(evidence['masking']).query('target == @target')
mask = (-100*mask.pivot(index='group',columns='session',values='delta')).reindex(columns=rats)
display(mask.assign(mean=mask.mean(axis=1)).round(1))
cohort = 'all'  # all or outseq
label = target  # group by odor or position
latent = {r['rat']: r for r in evidence['latents'] if r['target'] == target}
fig, axes = plt.subplots(1, len(rats), figsize=(14, 3), layout='constrained')
for ax, rat in zip(axes, rats, strict=True):
    r = latent[rat]
    points = [p for p in r['groups'] if p['cohort']==cohort and p['label']==label]
    ax.set(title=f'{rat} · {sum(p["n"] for p in points)} trials', xlabel='PC1')
    ax.axhline(0, color='0.85'); ax.axvline(0, color='0.85')
    for p in points:
        x,y = p['xy']
        ax.scatter(x,y,color=f'C{p["cls"]-1}',s=42)
        name = 'ABCDE'[p['cls']-1] if label == 'odor' else str(p['cls'])
        ax.annotate(f'{name} ({p["n"]})',(x,y),xytext=(3,3),textcoords='offset points',fontsize=8)
axes[0].set_ylabel('PC2');